# Complete Sentiment Analysis Pipeline
## EDA → Feature Engineering → Model Training → Hyperparameter Tuning → Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
from scipy.stats import uniform, randint

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("="*100)
print("COMPLETE SENTIMENT ANALYSIS PIPELINE - TARGET F1 > 0.9933")
print("="*100)

## PART 1: EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
print("\n" + "="*100)
print("PART 1: EXPLORATORY DATA ANALYSIS")
print("="*100)

train = pd.read_csv('Dataset/train.csv')
test = pd.read_csv('Dataset/test.csv')

print(f"\n1.1 Dataset Shape:")
print(f"  Train: {train.shape}")
print(f"  Test:  {test.shape}")

print(f"\n1.2 Column Info:")
print(train.info())

print(f"\n1.3 First Few Rows:")
print(train.head())

In [ ]:
print(f"\n1.4 Missing Values:")
print(f"\nTrain:")
print(train.isnull().sum())
print(f"\nTest:")
print(test.isnull().sum())

print(f"\n1.5 Rating Distribution:")
print(train['Rating'].value_counts().sort_index())
print(f"\nClass Balance:")
print(train['Rating'].value_counts(normalize=True).sort_index())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Rating distribution
train['Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color=['red', 'green'])
axes[0,0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Rating')
axes[0,0].set_ylabel('Count')

# Review length by rating
train['review_len'] = train['Review'].fillna('').astype(str).apply(len)
train.boxplot(column='review_len', by='Rating', ax=axes[0,1])
axes[0,1].set_title('Review Length by Rating', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Rating')
axes[0,1].set_ylabel('Review Length')

# Title length by rating
train['title_len'] = train['Review_Title'].fillna('').astype(str).apply(len)
train.boxplot(column='title_len', by='Rating', ax=axes[1,0])
axes[1,0].set_title('Title Length by Rating', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Rating')
axes[1,0].set_ylabel('Title Length')

# Word count by rating
train['word_count'] = train['Review'].fillna('').astype(str).apply(lambda x: len(x.split()))
train.boxplot(column='word_count', by='Rating', ax=axes[1,1])
axes[1,1].set_title('Word Count by Rating', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Rating')
axes[1,1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ EDA plots saved as 'eda_plots.png'")

In [ ]:
print(f"\n1.6 Statistical Summary by Rating:")
summary = train.groupby('Rating')[['review_len', 'title_len', 'word_count']].agg(['mean', 'median', 'std'])
print(summary)

print(f"\n1.7 Sample Reviews by Rating:")
print("\nNegative (Rating=0):")
print(train[train['Rating']==0][['Review_Title', 'Review']].head(3))
print("\nPositive (Rating=1):")
print(train[train['Rating']==1][['Review_Title', 'Review']].head(3))

## PART 2: TEXT PREPROCESSING & FEATURE ENGINEERING

In [ ]:
print("\n" + "="*100)
print("PART 2: TEXT PREPROCESSING & FEATURE ENGINEERING")
print("="*100)

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s!?.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("\n2.1 Cleaning text...")
train['text'] = (train['Review_Title'].fillna('') + ' ' + train['Review'].fillna('')).apply(clean_text)
test['text'] = (test['Review_Title'].fillna('') + ' ' + test['Review'].fillna('')).apply(clean_text)
train['title_clean'] = train['Review_Title'].fillna('').apply(clean_text)
test['title_clean'] = test['Review_Title'].fillna('').apply(clean_text)
train['review_clean'] = train['Review'].fillna('').apply(clean_text)
test['review_clean'] = test['Review'].fillna('').apply(clean_text)
print("✓ Text cleaning complete")

In [ ]:
print("\n2.2 Creating advanced features...")

sia = SentimentIntensityAnalyzer()

positive_words = {'excellent', 'amazing', 'great', 'love', 'perfect', 'best', 'awesome', 'fantastic', 
                  'wonderful', 'good', 'nice', 'happy', 'satisfied', 'recommend', 'quality', 'beautiful',
                  'superb', 'outstanding', 'brilliant', 'delightful', 'impressive', 'superior', 'exceptional'}

negative_words = {'bad', 'terrible', 'worst', 'poor', 'horrible', 'awful', 'disappointing', 'useless',
                  'waste', 'broken', 'defective', 'cheap', 'hate', 'never', 'return', 'refund', 'pathetic',
                  'disgusting', 'rubbish', 'garbage', 'fake', 'fraud', 'scam', 'junk'}

def extract_features(df):
    # VADER sentiment
    sentiment = df['text'].apply(lambda x: sia.polarity_scores(x))
    df['sent_compound'] = sentiment.apply(lambda x: x['compound'])
    df['sent_pos'] = sentiment.apply(lambda x: x['pos'])
    df['sent_neg'] = sentiment.apply(lambda x: x['neg'])
    df['sent_neu'] = sentiment.apply(lambda x: x['neu'])
    
    # Title & Review sentiment
    title_sent = df['title_clean'].apply(lambda x: sia.polarity_scores(x))
    df['title_compound'] = title_sent.apply(lambda x: x['compound'])
    review_sent = df['review_clean'].apply(lambda x: sia.polarity_scores(x))
    df['review_compound'] = review_sent.apply(lambda x: x['compound'])
    
    # Text statistics
    df['char_count'] = df['text'].apply(len)
    df['word_count'] = df['text'].apply(lambda x: len(x.split()))
    df['exclamation'] = df['Review'].fillna('').astype(str).apply(lambda x: x.count('!'))
    df['question'] = df['Review'].fillna('').astype(str).apply(lambda x: x.count('?'))
    
    # Positive/negative words
    df['pos_words'] = df['text'].apply(lambda x: sum(1 for w in x.split() if w in positive_words))
    df['neg_words'] = df['text'].apply(lambda x: sum(1 for w in x.split() if w in negative_words))
    df['pos_neg_ratio'] = (df['pos_words'] + 1) / (df['neg_words'] + 1)
    df['sentiment_score'] = df['pos_words'] - df['neg_words']
    
    # Interaction features
    df['compound_x_pos'] = df['sent_compound'] * df['pos_words']
    df['compound_x_neg'] = df['sent_compound'] * df['neg_words']
    
    return df

train = extract_features(train)
test = extract_features(test)

feature_cols = ['sent_compound', 'sent_pos', 'sent_neg', 'sent_neu', 'title_compound', 'review_compound',
                'char_count', 'word_count', 'exclamation', 'question', 'pos_words', 'neg_words', 
                'pos_neg_ratio', 'sentiment_score', 'compound_x_pos', 'compound_x_neg']

print(f"✓ Created {len(feature_cols)} features")
print(f"\nFeature list: {feature_cols}")

In [ ]:
print("\n2.3 Feature correlation with target...")
correlations = train[feature_cols + ['Rating']].corr()['Rating'].drop('Rating').sort_values(ascending=False)
print(correlations)

plt.figure(figsize=(10, 8))
correlations.plot(kind='barh', color='steelblue')
plt.title('Feature Correlation with Rating', fontsize=14, fontweight='bold')
plt.xlabel('Correlation')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Correlation plot saved as 'feature_correlation.png'")

In [ ]:
print("\n2.4 Creating TF-IDF features...")

# Word-level TF-IDF
tfidf_word = TfidfVectorizer(max_features=15000, ngram_range=(1, 3), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_word = tfidf_word.fit_transform(train['text'])
X_test_word = tfidf_word.transform(test['text'])
print(f"  Word TF-IDF: {X_train_word.shape}")

# Character-level TF-IDF
tfidf_char = TfidfVectorizer(max_features=8000, analyzer='char', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
X_train_char = tfidf_char.fit_transform(train['text'])
X_test_char = tfidf_char.transform(test['text'])
print(f"  Char TF-IDF: {X_train_char.shape}")

# Title TF-IDF
tfidf_title = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
X_train_title = tfidf_title.fit_transform(train['title_clean'])
X_test_title = tfidf_title.transform(test['title_clean'])
print(f"  Title TF-IDF: {X_train_title.shape}")

# Scale numerical features
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(train[feature_cols])
X_test_feat = scaler.transform(test[feature_cols])
print(f"  Numerical features: {X_train_feat.shape}")

# Combine all
X_train = hstack([X_train_word, X_train_char, X_train_title, X_train_feat])
X_test = hstack([X_test_word, X_test_char, X_test_title, X_test_feat])
y_train = train['Rating'].values

print(f"\n✓ Final feature matrix: {X_train.shape}")

## PART 3: MODEL TRAINING & HYPERPARAMETER TUNING

In [ ]:
print("\n" + "="*100)
print("PART 3: MODEL TRAINING & HYPERPARAMETER TUNING")
print("="*100)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_cv(model, X, y, name):
    scores = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        scores.append(f1)
    mean_f1 = np.mean(scores)
    std_f1 = np.std(scores)
    print(f"  {name:20s} CV F1: {mean_f1:.6f} (±{std_f1:.6f})")
    return mean_f1, std_f1

In [ ]:
print("\n3.1 Baseline Models (Default Parameters)...")

baseline_results = {}

models_baseline = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=100, random_state=42, verbose=0)
}

for name, model in models_baseline.items():
    mean_f1, std_f1 = evaluate_cv(model, X_train, y_train, name)
    baseline_results[name] = mean_f1

In [ ]:
print("\n3.2 Hyperparameter Tuning - Logistic Regression...")

lr_params = {'C': [1.0, 2.0, 3.0, 5.0, 7.0], 'class_weight': ['balanced', None], 'solver': ['saga', 'lbfgs']}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), lr_params, 
                       cv=3, scoring='f1', n_jobs=-1, verbose=1)
lr_grid.fit(X_train, y_train)
print(f"  Best params: {lr_grid.best_params_}")
print(f"  Best CV F1: {lr_grid.best_score_:.6f}")
best_lr = lr_grid.best_estimator_

In [ ]:
print("\n3.3 Hyperparameter Tuning - XGBoost...")

xgb_params = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.02, 0.03, 0.05],
    'max_depth': [7, 9, 11],
    'subsample': [0.85, 0.9],
    'colsample_bytree': [0.85, 0.9]
}
xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, eval_metric='logloss', tree_method='hist'), 
                                xgb_params, n_iter=20, cv=3, scoring='f1', n_jobs=-1, random_state=42, verbose=1)
xgb_random.fit(X_train, y_train)
print(f"  Best params: {xgb_random.best_params_}")
print(f"  Best CV F1: {xgb_random.best_score_:.6f}")
best_xgb = xgb_random.best_estimator_

In [ ]:
print("\n3.4 Hyperparameter Tuning - LightGBM...")

lgbm_params = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.02, 0.03, 0.05],
    'max_depth': [8, 10, 12],
    'num_leaves': [50, 70, 90],
    'subsample': [0.85, 0.9]
}
lgbm_random = RandomizedSearchCV(LGBMClassifier(random_state=42, verbose=-1), lgbm_params, 
                                 n_iter=20, cv=3, scoring='f1', n_jobs=-1, random_state=42, verbose=1)
lgbm_random.fit(X_train, y_train)
print(f"  Best params: {lgbm_random.best_params_}")
print(f"  Best CV F1: {lgbm_random.best_score_:.6f}")
best_lgbm = lgbm_random.best_estimator_

In [ ]:
print("\n3.5 Hyperparameter Tuning - CatBoost...")

catboost_params = {
    'iterations': [300, 500, 700],
    'learning_rate': [0.02, 0.03, 0.05],
    'depth': [7, 9, 11]
}
cat_random = RandomizedSearchCV(CatBoostClassifier(random_state=42, verbose=0), catboost_params, 
                                n_iter=15, cv=3, scoring='f1', n_jobs=-1, random_state=42, verbose=1)
cat_random.fit(X_train, y_train)
print(f"  Best params: {cat_random.best_params_}")
print(f"  Best CV F1: {cat_random.best_score_:.6f}")
best_cat = cat_random.best_estimator_

## PART 4: FINAL EVALUATION & ENSEMBLE

In [ ]:
print("\n" + "="*100)
print("PART 4: FINAL EVALUATION & ENSEMBLE")
print("="*100)

print("\n4.1 Evaluating tuned models with 5-fold CV...")

tuned_models = {
    'Logistic Regression': best_lr,
    'XGBoost': best_xgb,
    'LightGBM': best_lgbm,
    'CatBoost': best_cat
}

tuned_results = {}
for name, model in tuned_models.items():
    mean_f1, std_f1 = evaluate_cv(model, X_train, y_train, name)
    tuned_results[name] = mean_f1

In [ ]:
print("\n4.2 Model Comparison...")

comparison_df = pd.DataFrame({
    'Baseline F1': baseline_results,
    'Tuned F1': tuned_results,
    'Improvement': [tuned_results.get(k, 0) - baseline_results.get(k, 0) for k in tuned_results.keys()]
})
print(comparison_df.sort_values('Tuned F1', ascending=False))

plt.figure(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.35
plt.bar(x - width/2, comparison_df['Baseline F1'], width, label='Baseline', alpha=0.8)
plt.bar(x + width/2, comparison_df['Tuned F1'], width, label='Tuned', alpha=0.8)
plt.xlabel('Models')
plt.ylabel('F1-Score')
plt.title('Model Performance: Baseline vs Tuned', fontsize=14, fontweight='bold')
plt.xticks(x, comparison_df.index, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n✓ Comparison plot saved as 'model_comparison.png'")

In [ ]:
print("\n4.3 Training final ensemble...")

# Train all tuned models on full data
for name, model in tuned_models.items():
    print(f"  Training {name}...")
    model.fit(X_train, y_train)

# Generate predictions
test_preds = np.zeros((len(test), len(tuned_models)))
for i, (name, model) in enumerate(tuned_models.items()):
    test_preds[:, i] = model.predict(X_test)

# Weighted ensemble based on CV scores
weights = np.array([tuned_results[name] for name in tuned_models.keys()])
weights = weights / weights.sum()

print(f"\nEnsemble weights:")
for name, w in zip(tuned_models.keys(), weights):
    print(f"  {name:20s}: {w:.4f}")

final_pred = np.zeros(len(test))
for i, w in enumerate(weights):
    final_pred += w * test_preds[:, i]

final_pred = (final_pred >= 0.5).astype(int)
print("\n✓ Ensemble predictions generated")

## PART 5: RESULTS & SUBMISSION

In [ ]:
print("\n" + "="*100)
print("PART 5: RESULTS & SUBMISSION")
print("="*100)

submission = pd.DataFrame({'ID': test['ID'], 'Rating': final_pred})
submission.to_csv('submission.csv', index=False)

print("\n5.1 Submission Statistics:")
print(f"  Total predictions: {len(submission)}")
print(f"\n  Predicted distribution:")
print(submission['Rating'].value_counts().sort_index())
print(f"\n  Predicted proportions:")
print(submission['Rating'].value_counts(normalize=True).sort_index())

print(f"\n5.2 Expected Performance:")
print(f"  Best single model CV F1: {max(tuned_results.values()):.6f}")
print(f"  Expected ensemble F1: > {max(tuned_results.values()):.6f}")

print("\n" + "="*100)
print("✓ SUBMISSION FILE CREATED: submission.csv")
print("="*100)
print(f"\nTarget: F1 > 0.9933")
print(f"Expected: F1 ≈ {max(tuned_results.values()):.6f}")
print("\n" + "="*100)